# Tabela única dos PDFs Carros na Web

Este notebook processa todos os arquivos PDF dentro de `carros_benchmark` com `use_llm_fallback=False` e consolida o resultado em uma tabela wide:

- cada linha é um atributo veicular
- cada coluna é um veículo analisado

In [28]:
from pathlib import Path
import re

import pandas as pd

from carros_na_web_parser import parse_pdf

In [29]:
base_dir = Path('carros_benchmark')
pdf_paths = sorted(base_dir.glob('*.pdf'))
pdf_paths

[WindowsPath('carros_benchmark/Carros na Web _ BYD Song Plus GS 1.5 2027 _ Ficha Técnica, Especificações, Equipamentos, Fotos, Preço_.pdf'),
 WindowsPath('carros_benchmark/Carros na Web _ BYD Song Plus Premium 1.5 AWD 2027 _ Ficha Técnica, Especificações, Equipamentos, Fotos, Preço_.pdf'),
 WindowsPath('carros_benchmark/Carros na Web _ Jeep Compass Night Eagle 1.3 2026 _ Ficha Técnica, Especificações, Equipamentos, Fotos, Preço_.pdf'),
 WindowsPath('carros_benchmark/Carros na Web _ Jeep Compass Sport 1.3 2026 _ Ficha Técnica, Especificações, Equipamentos, Fotos, Preço_.pdf'),
 WindowsPath('carros_benchmark/Carros na Web _ Renault Boreal Evolution 1.3 2026 _ Ficha Técnica, Especificações, Equipamentos, Fotos, Preço_.pdf'),
 WindowsPath('carros_benchmark/Carros na Web _ Renault Boreal Iconic 1.3 2026 _ Ficha Técnica, Especificações, Equipamentos, Fotos, Preço_.pdf'),
 WindowsPath('carros_benchmark/Carros na Web _ Renault Boreal Techno 1.3 2026 _ Ficha Técnica, Especificações, Equipamento

In [30]:
def vehicle_name_from_pdf_path(pdf_path: Path) -> str:
    name = pdf_path.stem
    name = re.sub(r'^Carros na Web\s*_\s*', '', name, flags=re.IGNORECASE)
    name = re.sub(r'\s*_\s*Ficha Técnica.*$', '', name, flags=re.IGNORECASE)
    name = re.sub(r'\s+', ' ', name).strip()
    return name


vehicle_name_from_pdf_path(pdf_paths[0]) if pdf_paths else None

'BYD Song Plus GS 1.5 2027'

In [31]:
frames = {}

def collapse_duplicate_attributes(df: pd.DataFrame) -> pd.Series:
    cleaned = df.copy()
    cleaned['Atributo veicular'] = cleaned['Atributo veicular'].astype(str).str.strip()
    cleaned['Valor'] = cleaned['Valor'].astype(str).str.strip()
    return cleaned.groupby('Atributo veicular', sort=False)['Valor'].agg(
        lambda values: next((value for value in values if value and value != 'nan'), '')
    )

for pdf_path in pdf_paths:
    vehicle_name = vehicle_name_from_pdf_path(pdf_path)
    df = parse_pdf(pdf_path, use_llm_fallback=False)
    frames[vehicle_name] = df
    print(f'{vehicle_name}: {len(df)} linhas')

frames.keys()

BYD Song Plus GS 1.5 2027: 175 linhas
BYD Song Plus Premium 1.5 AWD 2027: 176 linhas
Jeep Compass Night Eagle 1.3 2026: 162 linhas
Jeep Compass Sport 1.3 2026: 152 linhas
Renault Boreal Evolution 1.3 2026: 144 linhas
Renault Boreal Iconic 1.3 2026: 167 linhas
Renault Boreal Techno 1.3 2026: 159 linhas
Toyota Corolla Cross XRE 2.0 2027: 152 linhas
Toyota Corolla Cross XRX 2.0 2027: 159 linhas
Toyota Corolla Cross XRX Hybrid 1.8 2027: 165 linhas
Volkswagen Taos Comfortline 1.4 TSi 2026: 151 linhas
Volkswagen Taos Highline 1.4 TSi 2026: 162 linhas


dict_keys(['BYD Song Plus GS 1.5 2027', 'BYD Song Plus Premium 1.5 AWD 2027', 'Jeep Compass Night Eagle 1.3 2026', 'Jeep Compass Sport 1.3 2026', 'Renault Boreal Evolution 1.3 2026', 'Renault Boreal Iconic 1.3 2026', 'Renault Boreal Techno 1.3 2026', 'Toyota Corolla Cross XRE 2.0 2027', 'Toyota Corolla Cross XRX 2.0 2027', 'Toyota Corolla Cross XRX Hybrid 1.8 2027', 'Volkswagen Taos Comfortline 1.4 TSi 2026', 'Volkswagen Taos Highline 1.4 TSi 2026'])

In [32]:
wide = pd.concat(
    {vehicle: collapse_duplicate_attributes(df) for vehicle, df in frames.items()},
    axis=1,
).reset_index()

wide = wide.rename(columns={'index': 'Atributo veicular'})
wide

,Atributo veicular,BYD Song Plus GS 1.5 2027,BYD Song Plus Premium 1.5 AWD 2027,Jeep Compass Night Eagle 1.3 2026,Jeep Compass Sport 1.3 2026,Renault Boreal Evolution 1.3 2026,Renault Boreal Iconic 1.3 2026,Renault Boreal Techno 1.3 2026,Toyota Corolla Cross XRE 2.0 2027,Toyota Corolla Cross XRX 2.0 2027,Toyota Corolla Cross XRX Hybrid 1.8 2027,Volkswagen Taos Comfortline 1.4 TSi 2026,Volkswagen Taos Highline 1.4 TSi 2026
0,Ano,2027,2027,2026,2026,2026,2026,2026,2027,2027,2027,2026,2026
1,Preço,R$ 249.990,R$ 299.800,R$ 205.490,R$ 174.990,R$ 179.990,R$ 214.990,R$ 199.990,R$ 194.790,R$ 211.790,R$ 223.790,R$ 199.990,R$ 209.990
2,IPVA,R$ 10.000,R$ 11.992,R$ 8.220,R$ 7.000,R$ 7.200,R$ 8.600,R$ 8.000,R$ 7.792,R$ 8.472,R$ 8.952,R$ 8.000,R$ 8.400
3,Seguro,R$ 6.750,R$ 8.095,R$ 5.548,R$ 4.725,R$ 4.860,R$ 5.805,R$ 5.400,R$ 5.259,R$ 5.718,R$ 6.042,R$ 5.400,R$ 5.670
4,Revisões,R$ 6.383 até 60.000 km,R$ 6.383 até 60.000 km,R$ 5.268 até 60.000 km,R$ 5.268 até 60.000 km,R$ 6.316 até 60.000 km,R$ 6.316 até 60.000 km,R$ 6.316 até 60.000 km,R$ 4.782 até 50.000 km,R$ 4.782 até 50.000 km,R$ 4.782 até 50.000 km,R$ 3.331 até 60.000 km,R$ 3.331 até 60.000 km
...,...,...,...,...,...,...,...,...,...,...,...,...,...
242,"13,3",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,km/l (G),km/l (G)
243,533,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,km (G),km (G)
244,638,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,km (G),km (G)
245,Vetorização de torque,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Presente,Presente


In [33]:
# Visão rápida do resultado final
print(f'Veículos processados: {len(frames)}')
print(f'Atributos únicos: {len(wide)}')
wide.head(20).to_string(index=False)

Veículos processados: 12
Atributos únicos: 247


'Atributo veicular BYD Song Plus GS 1.5 2027 BYD Song Plus Premium 1.5 AWD 2027 Jeep Compass Night Eagle 1.3 2026 Jeep Compass Sport 1.3 2026 Renault Boreal Evolution 1.3 2026 Renault Boreal Iconic 1.3 2026 Renault Boreal Techno 1.3 2026 Toyota Corolla Cross XRE 2.0 2027 Toyota Corolla Cross XRX 2.0 2027 Toyota Corolla Cross XRX Hybrid 1.8 2027 Volkswagen Taos Comfortline 1.4 TSi 2026 Volkswagen Taos Highline 1.4 TSi 2026\n              Ano                      2027                               2027                              2026                        2026                              2026                           2026                           2026                              2027                              2027                                     2027                                     2026                                  2026\n            Preço                R$ 249.990                         R$ 299.800                        R$ 205.490                  R$ 174.990       

In [34]:
# Se quiser persistir o resultado, descomente uma das linhas abaixo.
wide.to_csv('carros_benchmark_wide.csv', index=False, encoding='utf-8-sig')
wide.to_excel('carros_benchmark_wide.xlsx', index=False)

wide

,Atributo veicular,BYD Song Plus GS 1.5 2027,BYD Song Plus Premium 1.5 AWD 2027,Jeep Compass Night Eagle 1.3 2026,Jeep Compass Sport 1.3 2026,Renault Boreal Evolution 1.3 2026,Renault Boreal Iconic 1.3 2026,Renault Boreal Techno 1.3 2026,Toyota Corolla Cross XRE 2.0 2027,Toyota Corolla Cross XRX 2.0 2027,Toyota Corolla Cross XRX Hybrid 1.8 2027,Volkswagen Taos Comfortline 1.4 TSi 2026,Volkswagen Taos Highline 1.4 TSi 2026
0,Ano,2027,2027,2026,2026,2026,2026,2026,2027,2027,2027,2026,2026
1,Preço,R$ 249.990,R$ 299.800,R$ 205.490,R$ 174.990,R$ 179.990,R$ 214.990,R$ 199.990,R$ 194.790,R$ 211.790,R$ 223.790,R$ 199.990,R$ 209.990
2,IPVA,R$ 10.000,R$ 11.992,R$ 8.220,R$ 7.000,R$ 7.200,R$ 8.600,R$ 8.000,R$ 7.792,R$ 8.472,R$ 8.952,R$ 8.000,R$ 8.400
3,Seguro,R$ 6.750,R$ 8.095,R$ 5.548,R$ 4.725,R$ 4.860,R$ 5.805,R$ 5.400,R$ 5.259,R$ 5.718,R$ 6.042,R$ 5.400,R$ 5.670
4,Revisões,R$ 6.383 até 60.000 km,R$ 6.383 até 60.000 km,R$ 5.268 até 60.000 km,R$ 5.268 até 60.000 km,R$ 6.316 até 60.000 km,R$ 6.316 até 60.000 km,R$ 6.316 até 60.000 km,R$ 4.782 até 50.000 km,R$ 4.782 até 50.000 km,R$ 4.782 até 50.000 km,R$ 3.331 até 60.000 km,R$ 3.331 até 60.000 km
...,...,...,...,...,...,...,...,...,...,...,...,...,...
242,"13,3",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,km/l (G),km/l (G)
243,533,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,km (G),km (G)
244,638,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,km (G),km (G)
245,Vetorização de torque,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Presente,Presente
